In [1]:
import os

BASE = "CSL7110-Assignment2-M25DE1006"

folders = [
    f"{BASE}/data",
    f"{BASE}/notebooks",
    f"{BASE}/movielens",
    f"{BASE}/results/screenshots",
    f"{BASE}/results/tables",
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Created folders:")
for f in folders:
    print(" -", f)

Created folders:
 - CSL7110-Assignment2-M25DE1006/data
 - CSL7110-Assignment2-M25DE1006/notebooks
 - CSL7110-Assignment2-M25DE1006/movielens
 - CSL7110-Assignment2-M25DE1006/results/screenshots
 - CSL7110-Assignment2-M25DE1006/results/tables


In [2]:
from google.colab import files
import shutil, os

BASE = "CSL7110-Assignment2-M25DE1006"
uploaded = files.upload()  # upload D1.txt..D4.txt

for fname in uploaded.keys():
    shutil.move(fname, f"{BASE}/data/{fname}")

print("Files now in data/:")
print(os.listdir(f"{BASE}/data"))

Saving D1.txt to D1.txt
Saving D2.txt to D2.txt
Saving D3.txt to D3.txt
Saving D4.txt to D4.txt
Files now in data/:
['D3.txt', 'D2.txt', 'D1.txt', 'D4.txt']


In [3]:
import datetime, platform, sys

BASE = "CSL7110-Assignment2-M25DE1006"
meta = f"""
Run timestamp (IST approx): {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Python: {sys.version}
Platform: {platform.platform()}
"""

with open(f"{BASE}/results/run_metadata.txt", "w") as f:
    f.write(meta.strip())

print(meta)


Run timestamp (IST approx): 2026-03-05 17:06:31
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35



In [4]:
import os, re
from pathlib import Path

BASE = "CSL7110-Assignment2-M25DE1006"
DATA_DIR = Path(BASE) / "data"

doc_files = ["D1.txt", "D2.txt", "D3.txt", "D4.txt"]
docs = {}

for f in doc_files:
    text = (DATA_DIR / f).read_text(encoding="utf-8", errors="ignore")
    text = text.lower()  # assignment says lowercase letters + space
    # normalize whitespace (keeps spaces as a character, but avoids weird runs)
    text = re.sub(r"\s+", " ", text).strip()
    docs[f.replace(".txt","")] = text

# quick sanity preview
for k,v in docs.items():
    print(k, "| length:", len(v), "| preview:", v[:90], "...")

D1 | length: 1749 | preview: apple ceo tim cook is spending some time in canada this week and yesterday he attended a h ...
D2 | length: 1747 | preview: apple ceo tim cook is spending some time in canada this week and yesterday attended a hock ...
D3 | length: 2132 | preview: as part of his one day tour of canada yesterday tim cook offered an interview to the finan ...
D4 | length: 1435 | preview: president trump who warned as a candidate about the false song of globalism is marching st ...


In [5]:
def char_kgrams(text: str, k: int) -> set:
    # includes space as a character naturally
    if len(text) < k:
        return set()
    return {text[i:i+k] for i in range(len(text) - k + 1)}

def word_kgrams(text: str, k: int) -> set:
    # split into words; ignore empty tokens
    words = [w for w in text.split(" ") if w != ""]
    if len(words) < k:
        return set()
    return {" ".join(words[i:i+k]) for i in range(len(words) - k + 1)}

def jaccard(A: set, B: set) -> float:
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

In [6]:
kgram_sets = {}

for docname, text in docs.items():
    kgram_sets[(docname, "char_2")] = char_kgrams(text, 2)
    kgram_sets[(docname, "char_3")] = char_kgrams(text, 3)
    kgram_sets[(docname, "word_2")] = word_kgrams(text, 2)

# show counts (this is good for observations in the report)
for docname in docs.keys():
    c2 = len(kgram_sets[(docname, "char_2")])
    c3 = len(kgram_sets[(docname, "char_3")])
    w2 = len(kgram_sets[(docname, "word_2")])
    print(f"{docname}: char_2={c2}, char_3={c3}, word_2={w2}")


D1: char_2=263, char_3=765, word_2=279
D2: char_2=262, char_3=762, word_2=278
D3: char_2=269, char_3=828, word_2=337
D4: char_2=255, char_3=698, word_2=232


In [7]:
import pandas as pd
import itertools

docs_list = ["D1","D2","D3","D4"]
pairs = list(itertools.combinations(docs_list,2))

gram_types = ["char_2","char_3","word_2"]

results = []

for gram in gram_types:
    for d1,d2 in pairs:
        A = kgram_sets[(d1,gram)]
        B = kgram_sets[(d2,gram)]

        sim = jaccard(A,B)

        results.append({
            "kgram_type": gram,
            "doc1": d1,
            "doc2": d2,
            "jaccard_similarity": round(sim,4)
        })

df_jaccard = pd.DataFrame(results)

df_jaccard

,kgram_type,doc1,doc2,jaccard_similarity
0,char_2,D1,D2,0.9811
1,char_2,D1,D3,0.8157
2,char_2,D1,D4,0.6444
3,char_2,D2,D3,0.8000
4,char_2,D2,D4,0.6413
5,char_2,D3,D4,0.6530
6,char_3,D1,D2,0.9780
7,char_3,D1,D3,0.5804
8,char_3,D1,D4,0.3051
9,char_3,D2,D3,0.5680


In [8]:
df_jaccard.to_csv(
    "CSL7110-Assignment2-M25DE1006/results/tables/jaccard_results.csv",
    index=False
)

print("Saved results to results/tables/jaccard_results.csv")

Saved results to results/tables/jaccard_results.csv


In [9]:
import hashlib

# Collect all 3-grams from all documents
all_3grams = set()

for doc in docs.keys():
    all_3grams |= kgram_sets[(doc,"char_3")]

# Convert to integer IDs
def gram_to_int(g):
    return int(hashlib.md5(g.encode()).hexdigest(),16)

gram_ids = {g: gram_to_int(g) for g in all_3grams}

print("Total unique 3-grams:", len(gram_ids))

Total unique 3-grams: 1301


In [10]:
D1_set = {gram_ids[g] for g in kgram_sets[("D1","char_3")]}
D2_set = {gram_ids[g] for g in kgram_sets[("D2","char_3")]}

print("D1 grams:",len(D1_set))
print("D2 grams:",len(D2_set))

D1 grams: 765
D2 grams: 762


In [11]:
import random

def generate_hash_funcs(t, m=20000):

    p = 2147483647  # large prime
    funcs = []

    for _ in range(t):
        a = random.randint(1,p-1)
        b = random.randint(0,p-1)

        funcs.append(lambda x, a=a, b=b: ((a*x + b) % p) % m)

    return funcs

In [12]:
def minhash_signature(doc_set, hash_funcs):

    sig = []

    for h in hash_funcs:
        values = [h(x) for x in doc_set]
        sig.append(min(values))

    return sig


In [13]:
import pandas as pd

t_values = [20,60,150,300,600]

results = []

for t in t_values:

    hash_funcs = generate_hash_funcs(t)

    sig1 = minhash_signature(D1_set, hash_funcs)
    sig2 = minhash_signature(D2_set, hash_funcs)

    matches = sum(1 for a,b in zip(sig1,sig2) if a==b)

    approx_jaccard = matches / t

    results.append({
        "t_hash_functions":t,
        "approx_jaccard":round(approx_jaccard,4)
    })

df_minhash = pd.DataFrame(results)

df_minhash

,t_hash_functions,approx_jaccard
0,20,0.950
1,60,1.000
2,150,0.980
3,300,0.970
4,600,0.975


In [14]:
df_minhash.to_csv(
"CSL7110-Assignment2-M25DE1006/results/tables/minhash_estimates.csv",
index=False)

print("Saved MinHash results")

Saved MinHash results


In [15]:
import time
import pandas as pd

true_jaccard = df_jaccard[
    (df_jaccard.kgram_type=="char_3") &
    (df_jaccard.doc1=="D1") &
    (df_jaccard.doc2=="D2")
]["jaccard_similarity"].values[0]

t_values = [20,60,150,300,600]

analysis = []

for t in t_values:

    start = time.time()

    hash_funcs = generate_hash_funcs(t)

    sig1 = minhash_signature(D1_set, hash_funcs)
    sig2 = minhash_signature(D2_set, hash_funcs)

    matches = sum(1 for a,b in zip(sig1,sig2) if a==b)

    approx = matches/t

    runtime = time.time()-start

    error = abs(true_jaccard-approx)

    analysis.append({
        "t":t,
        "approx_jaccard":round(approx,4),
        "true_jaccard":true_jaccard,
        "error":round(error,4),
        "runtime_seconds":round(runtime,4)
    })

df_analysis = pd.DataFrame(analysis)

df_analysis

,t,approx_jaccard,true_jaccard,error,runtime_seconds
0,20,1.0000,0.978,0.0220,0.0258
1,60,0.9667,0.978,0.0113,0.0724
2,150,1.0000,0.978,0.0220,0.3238
3,300,0.9833,0.978,0.0053,1.1974
4,600,0.9750,0.978,0.0030,0.7212


In [16]:
import time
import pandas as pd

true_jaccard = df_jaccard[
    (df_jaccard.kgram_type=="char_3") &
    (df_jaccard.doc1=="D1") &
    (df_jaccard.doc2=="D2")
]["jaccard_similarity"].values[0]

t_values = [20,60,150,300,600]

analysis = []

for t in t_values:

    start = time.time()

    hash_funcs = generate_hash_funcs(t)

    sig1 = minhash_signature(D1_set, hash_funcs)
    sig2 = minhash_signature(D2_set, hash_funcs)

    matches = sum(1 for a,b in zip(sig1,sig2) if a==b)

    approx = matches/t

    runtime = time.time()-start

    error = abs(true_jaccard-approx)

    analysis.append({
        "t":t,
        "approx_jaccard":round(approx,4),
        "true_jaccard":true_jaccard,
        "error":round(error,4),
        "runtime_seconds":round(runtime,4)
    })

df_analysis = pd.DataFrame(analysis)

df_analysis

,t,approx_jaccard,true_jaccard,error,runtime_seconds
0,20,1.0000,0.978,0.0220,0.0942
1,60,0.9833,0.978,0.0053,0.0428
2,150,0.9800,0.978,0.0020,0.1771
3,300,0.9767,0.978,0.0013,0.5460
4,600,0.9817,0.978,0.0037,0.7297


In [17]:
df_analysis.to_csv(
"CSL7110-Assignment2-M25DE1006/results/tables/minhash_runtime_analysis.csv",
index=False
)

In [19]:
df_analysis

,t,approx_jaccard,true_jaccard,error,runtime_seconds
0,20,1.0000,0.978,0.0220,0.0942
1,60,0.9833,0.978,0.0053,0.0428
2,150,0.9800,0.978,0.0020,0.1771
3,300,0.9767,0.978,0.0013,0.5460
4,600,0.9817,0.978,0.0037,0.7297


In [20]:
import pandas as pd

t = 160
tau = 0.7

candidates = []

for r in range(2,21):

    if t % r == 0:
        b = t // r

        threshold = (1/b)**(1/r)

        candidates.append({
            "rows_per_band_r": r,
            "bands_b": b,
            "approx_threshold": round(threshold,3)
        })

df_lsh_params = pd.DataFrame(candidates)

df_lsh_params

,rows_per_band_r,bands_b,approx_threshold
0,2,80,0.112
1,4,40,0.398
2,5,32,0.500
3,8,20,0.688
4,10,16,0.758
5,16,10,0.866
6,20,8,0.901


In [21]:
import pandas as pd

r = 8
b = 20

pairs = [
("D1","D2"),
("D1","D3"),
("D1","D4"),
("D2","D3"),
("D2","D4"),
("D3","D4")
]

prob_results = []

for d1,d2 in pairs:

    s = df_jaccard[
        (df_jaccard.kgram_type=="char_3") &
        (df_jaccard.doc1==d1) &
        (df_jaccard.doc2==d2)
    ]["jaccard_similarity"].values[0]

    prob = 1 - (1 - (s**r))**b

    prob_results.append({
        "doc1":d1,
        "doc2":d2,
        "similarity_s":round(s,4),
        "candidate_probability":round(prob,4)
    })

df_lsh_prob = pd.DataFrame(prob_results)

df_lsh_prob

,doc1,doc2,similarity_s,candidate_probability
0,D1,D2,0.9780,1.0000
1,D1,D3,0.5804,0.2283
2,D1,D4,0.3051,0.0015
3,D2,D3,0.5680,0.1958
4,D2,D4,0.3059,0.0015
5,D3,D4,0.3121,0.0018


In [22]:
df_lsh_prob


,doc1,doc2,similarity_s,candidate_probability
0,D1,D2,0.9780,1.0000
1,D1,D3,0.5804,0.2283
2,D1,D4,0.3051,0.0015
3,D2,D3,0.5680,0.1958
4,D2,D4,0.3059,0.0015
5,D3,D4,0.3121,0.0018


In [23]:
import os
import urllib.request
import zipfile

BASE = "CSL7110-Assignment2-M25DE1006"
DATA_DIR = f"{BASE}/movielens"

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = f"{DATA_DIR}/ml-100k.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Dataset downloaded and extracted")

Dataset downloaded and extracted


In [24]:
import pandas as pd

ratings = pd.read_csv(
f"{DATA_DIR}/ml-100k/u.data",
sep="\t",
names=["user","movie","rating","timestamp"]
)

ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [25]:
user_movies = ratings.groupby("user")["movie"].apply(set)

print("Total users:", len(user_movies))

Total users: 943


In [26]:
import itertools

exact_pairs = []

users = list(user_movies.index)

for u1,u2 in itertools.combinations(users,2):

    A = user_movies[u1]
    B = user_movies[u2]

    sim = len(A & B) / len(A | B)

    if sim >= 0.5:
        exact_pairs.append((u1,u2,sim))

len(exact_pairs)

10

In [27]:
all_movies = ratings["movie"].unique()

movie_to_int = {m:i for i,m in enumerate(all_movies)}

user_sets = {}

for user,movies in user_movies.items():
    user_sets[user] = {movie_to_int[m] for m in movies}

print("Prepared user sets:", len(user_sets))

Prepared user sets: 943


In [28]:
import random

def generate_hash_funcs(t, m=20000):

    p = 2147483647
    funcs = []

    for _ in range(t):

        a = random.randint(1,p-1)
        b = random.randint(0,p-1)

        funcs.append(lambda x, a=a, b=b: ((a*x+b)%p)%m)

    return funcs

In [29]:
def user_signature(user_set, hash_funcs):

    sig = []

    for h in hash_funcs:
        sig.append(min(h(x) for x in user_set))

    return sig

In [31]:
import itertools

hash_sizes = [50,100,200]

results = []

for t in hash_sizes:

    hash_funcs = generate_hash_funcs(t)

    signatures = {}

    for u in users:
        signatures[u] = user_signature(user_sets[u], hash_funcs)

    approx_pairs = []

    for u1,u2 in itertools.combinations(users,2):

        sig1 = signatures[u1]
        sig2 = signatures[u2]

        sim = sum(1 for a,b in zip(sig1,sig2) if a==b)/t

        if sim >= 0.5:
            approx_pairs.append((u1,u2))

    approx_set = set(tuple(sorted(p)) for p in approx_pairs)
    exact_set = set(tuple(sorted((u1,u2))) for u1,u2,_ in exact_pairs)

    false_pos = len(approx_set - exact_set)
    false_neg = len(exact_set - approx_set)

    results.append({
        "hash_functions":t,
        "false_positives":false_pos,
        "false_negatives":false_neg
    })

df_movielens = pd.DataFrame(results)

df_movielens

,hash_functions,false_positives,false_negatives
0,50,206,1
1,100,40,1
2,200,9,1


In [32]:
df_movielens

,hash_functions,false_positives,false_negatives
0,50,206,1
1,100,40,1
2,200,9,1


In [33]:
def lsh_candidates(signatures, r, b):

    buckets = {}
    candidates = set()

    for user, sig in signatures.items():

        for band in range(b):

            start = band*r
            end = start+r

            band_sig = tuple(sig[start:end])

            key = (band, band_sig)

            if key not in buckets:
                buckets[key] = []

            buckets[key].append(user)

    for bucket_users in buckets.values():

        if len(bucket_users) > 1:

            for u1,u2 in itertools.combinations(bucket_users,2):
                candidates.add(tuple(sorted((u1,u2))))

    return candidates


In [34]:
configs = [
(50,5,10),
(100,5,20),
(200,5,40),
(200,10,20)
]

lsh_results = []

for t,r,b in configs:

    hash_funcs = generate_hash_funcs(t)

    signatures = {}

    for u in users:
        signatures[u] = user_signature(user_sets[u], hash_funcs)

    cand_pairs = lsh_candidates(signatures,r,b)

    approx_pairs = []

    for u1,u2 in cand_pairs:

        sig1 = signatures[u1]
        sig2 = signatures[u2]

        sim = sum(1 for a,b in zip(sig1,sig2) if a==b)/t

        if sim >= 0.6:
            approx_pairs.append((u1,u2))

    approx_set = set(tuple(sorted(p)) for p in approx_pairs)
    exact_set = set(tuple(sorted((u1,u2))) for u1,u2,_ in exact_pairs)

    false_pos = len(approx_set - exact_set)
    false_neg = len(exact_set - approx_set)

    lsh_results.append({
        "hash_functions":t,
        "r":r,
        "b":b,
        "false_positives":false_pos,
        "false_negatives":false_neg
    })

df_lsh_movielens = pd.DataFrame(lsh_results)

df_lsh_movielens

,hash_functions,r,b,false_positives,false_negatives
0,50,5,10,5,7
1,100,5,20,0,7
2,200,5,40,0,7
3,200,10,20,0,8


In [36]:
df_lsh_movielens


,hash_functions,r,b,false_positives,false_negatives
0,50,5,10,5,7
1,100,5,20,0,7
2,200,5,40,0,7
3,200,10,20,0,8


In [37]:
df_lsh_movielens.to_csv(
    "CSL7110-Assignment2-M25DE1006/results/tables/lsh_movielens_results_tau0p6.csv",
    index=False
)
print("Saved: results/tables/lsh_movielens_results_tau0p6.csv")

Saved: results/tables/lsh_movielens_results_tau0p6.csv


In [38]:
threshold = 0.8

lsh_results_08 = []

for t,r,b in configs:

    hash_funcs = generate_hash_funcs(t)

    signatures = {}
    for u in users:
        signatures[u] = user_signature(user_sets[u], hash_funcs)

    cand_pairs = lsh_candidates(signatures, r, b)

    approx_pairs = []
    for u1,u2 in cand_pairs:

        sig1 = signatures[u1]
        sig2 = signatures[u2]

        sim = sum(1 for a,bv in zip(sig1,sig2) if a==bv)/t

        if sim >= threshold:
            approx_pairs.append((u1,u2))

    approx_set = set(tuple(sorted(p)) for p in approx_pairs)

    # exact set at 0.8 (new ground truth)
    exact_pairs_08 = []
    for u1,u2 in itertools.combinations(users,2):
        A = user_movies[u1]
        B = user_movies[u2]
        sim_exact = len(A & B) / len(A | B)
        if sim_exact >= threshold:
            exact_pairs_08.append((u1,u2))

    exact_set_08 = set(tuple(sorted(p)) for p in exact_pairs_08)

    false_pos = len(approx_set - exact_set_08)
    false_neg = len(exact_set_08 - approx_set)

    lsh_results_08.append({
        "hash_functions":t,
        "r":r,
        "b":b,
        "false_positives":false_pos,
        "false_negatives":false_neg,
        "exact_pairs_count": len(exact_set_08),
        "predicted_pairs_count": len(approx_set)
    })

df_lsh_movielens_08 = pd.DataFrame(lsh_results_08)

df_lsh_movielens_08

,hash_functions,r,b,false_positives,false_negatives,exact_pairs_count,predicted_pairs_count
0,50,5,10,0,0,1,1
1,100,5,20,0,0,1,1
2,200,5,40,0,0,1,1
3,200,10,20,0,0,1,1


In [39]:
df_lsh_movielens_08.to_csv(
    "CSL7110-Assignment2-M25DE1006/results/tables/lsh_movielens_results_tau0p8.csv",
    index=False
)
print("Saved: results/tables/lsh_movielens_results_tau0p8.csv")

Saved: results/tables/lsh_movielens_results_tau0p8.csv


In [40]:
!zip -r CSL7110-Assignment2-M25DE1006.zip CSL7110-Assignment2-M25DE1006

  adding: CSL7110-Assignment2-M25DE1006/ (stored 0%)
  adding: CSL7110-Assignment2-M25DE1006/data/ (stored 0%)
  adding: CSL7110-Assignment2-M25DE1006/data/D3.txt (deflated 56%)
  adding: CSL7110-Assignment2-M25DE1006/data/D2.txt (deflated 54%)
  adding: CSL7110-Assignment2-M25DE1006/data/D1.txt (deflated 53%)
  adding: CSL7110-Assignment2-M25DE1006/data/D4.txt (deflated 53%)
  adding: CSL7110-Assignment2-M25DE1006/notebooks/ (stored 0%)
  adding: CSL7110-Assignment2-M25DE1006/results/ (stored 0%)
  adding: CSL7110-Assignment2-M25DE1006/results/run_metadata.txt (deflated 8%)
  adding: CSL7110-Assignment2-M25DE1006/results/tables/ (stored 0%)
  adding: CSL7110-Assignment2-M25DE1006/results/tables/lsh_movielens_results_tau0p6.csv (deflated 27%)
  adding: CSL7110-Assignment2-M25DE1006/results/tables/minhash_runtime_analysis.csv (deflated 41%)
  adding: CSL7110-Assignment2-M25DE1006/results/tables/minhash_estimates.csv (deflated 13%)
  adding: CSL7110-Assignment2-M25DE1006/results/tables/j

In [42]:
from google.colab import files
files.download("CSL7110-Assignment2-M25DE1006.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>